# 🤖 まず自分で分析 → 後半で Gemini に頼んでみる（教育データ編）

**所要時間：約15分** ／ 対象：Colab に少し慣れた人（はじめての人はノート A から）

このノートでは、**カリフォルニアの学校データ（CASchools）** を題材に、
**前半：自分の手でデータ分析（EDA）** を体験し、
**後半：Colab の Gemini（データサイエンス・エージェント）に「分析して」と頼む** 流れを試します。

> ℹ️ **Gemini に頼む機能は、アカウントの種別・地域・組織の設定によっては表示されない／使えないことがあります。**
> そのため本ノートは、**まず誰でもできる「自分で分析」を先に** 行い、**後半で（使える人は）Gemini を試す** 構成にしています。
> Gemini が使えなくても、今日の学びは前半だけで完結します。

> ⚠️ **大事な前提**：AI の出力は **もっともらしく間違える** ことがあります。**必ず自分で検証**してください。


## 0. 準備（ライブラリと日本語フォント）

In [ ]:
!pip install -q japanize-matplotlib
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
pd.set_option('display.max_columns', 50)
print('準備OK ✅')

## 1. データを用意する（/content に保存）

今回の教育データ **CASchools**（カリフォルニアの 420 学区の学校データ）を読み込み、
Gemini が読めるように **`ca_schools.csv` として保存** します。

主な列の意味：

| 列 | 意味 |
|---|---|
| `students` / `teachers` | 生徒数 / 教員数 |
| `lunch` | 給食補助の対象になる生徒の割合（％）＝**家庭の経済的な厳しさの目安** |
| `expenditure` | 生徒 1 人あたりの **支出（教育費）** |
| `income` | 地域の平均所得 |
| `english` | 英語学習者（英語が母語でない）の割合（％） |
| `read` / `math` | **読解 / 数学のテスト平均点** |

In [ ]:
url = 'https://vincentarelbundock.github.io/Rdatasets/csv/AER/CASchools.csv'
df = pd.read_csv(url)
df = df.drop(columns=[c for c in df.columns if c.lower() == 'rownames'])

# 学力スコア（読解と数学の平均）と 生徒/教員比 を追加
df['score'] = (df['read'] + df['math']) / 2
df['str']   = df['students'] / df['teachers']   # student-teacher ratio

# Gemini が読めるように /content に保存
df.to_csv('ca_schools.csv', index=False)
print('保存しました → ca_schools.csv（', len(df), '行 )')
df.head()

## 2. まず自分で分析してみる（EDA）

データ分析は特別な魔法ではありません。下のセルを上から実行すると、
**基本的な EDA（探索的データ分析）** が体験できます。まずは全体像から。
（このあと後半で、同じことを Gemini にも頼んでみます。）


In [ ]:
# 全体像：行数・列、要約統計
print('形：', df.shape)
cols = ['students', 'teachers', 'str', 'lunch', 'expenditure', 'income', 'english', 'score']
df[cols].describe().round(1)

### 学力スコアに効くのは何？ ── 相関を見る

`score`（テスト平均点）と、各要因の **相関**（一緒に動く強さ。+1〜−1）を見ます。
プラスなら「一方が高いほど score も高い」、マイナスなら「一方が高いほど score は低い」。

In [ ]:
corr = df[cols].corr()['score'].drop('score').sort_values()
print('score との相関（小さい＝強い負の相関）:')
print(corr.round(3).to_string())

# 横棒グラフで可視化
plt.figure(figsize=(7, 3.6))
colors = ['#1A6BB0' if v < 0 else '#C8611C' for v in corr.values]
plt.barh(corr.index, corr.values, color=colors)
plt.axvline(0, color='#888', lw=1)
plt.title('テスト平均点 score との相関')
plt.xlabel('相関係数（−1〜+1）')
plt.tight_layout(); plt.show()

In [ ]:
# 散布図で2つの『効きそうな要因』を見比べる：給食補助率 vs 支出
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].scatter(df['lunch'], df['score'], s=18, alpha=0.6, color='#1A6BB0')
ax[0].set_title('給食補助率（貧困の目安）と 学力')
ax[0].set_xlabel('給食補助の対象生徒の割合 (%)')
ax[0].set_ylabel('テスト平均点 score')

ax[1].scatter(df['expenditure'], df['score'], s=18, alpha=0.6, color='#C8611C')
ax[1].set_title('生徒1人あたり支出 と 学力')
ax[1].set_xlabel('生徒1人あたり支出 ($)')
ax[1].set_ylabel('テスト平均点 score')

plt.tight_layout(); plt.show()

## 3. 結果をどう解釈する？（ここが一番大事）

このデータでは、よく次のような傾向が見えます（実行結果で確かめてください）：

- **給食補助率（＝家庭の経済的な厳しさ）が高い学区ほど、テスト平均点は低い**（強い負の相関）
- 一方で、**生徒1人あたりの支出と学力の相関は意外に弱い**

➡️ 「**お金をかければ点が上がる**」という直感に反して、**家庭の経済状況の影響がずっと大きい** ——
これは教育の研究でよく知られた、考えさせられる発見です。

> 🧠 **AI に頼んでも、最後の解釈は人間の仕事**。相関は「因果（原因→結果）」とは限りません。
> 「支出が無意味」ではなく「単純な相関だけでは語れない」と読むのが正しい姿勢です。

### やってみよう（発展）── 自分で／このあとの Gemini でも
- `english`（英語学習者の割合）と score の関係を散布図にしてみる
- `str`（生徒/教員比）が小さい（少人数）と score は高い？ を確かめる
- `score` を `lunch` と `income` から予測する回帰モデルを作る → ノート C と比べる


## 4.（発展）Gemini に分析を頼む — 使える人は試してみよう

ここからは **発展** です。Colab の **Gemini（データサイエンス・エージェント）** に、
**「分析して」と日本語で頼むだけ** で、前半と同じような分析ノートを自動で作れます。

> ℹ️ **この機能はアカウント・地域・組織の設定で表示されないことがあります。**
> パネルやアイコンが出ない場合は、無理に探さなくてOK（前半の「自分で分析」で今日の目標は達成しています）。

### やり方
1. 画面 **右側の Gemini パネル**（または下部の ✦ アイコン）を開く
2. すでに保存した **`ca_schools.csv`** を指定（左の 📁 にあります）
3. **やりたいことを日本語で書く**（下の例を貼ってOK）
4. Gemini が **手順（プラン）を立てて** コードを提案 → **「承認して実行」** で取り込む

### 頼み方の例（コピーして使えます）
> 📋 **`ca_schools.csv` を分析して。まず全体像（行数・列・要約統計）を出し、欠損を確認して。
> そのうえで、テストの平均点 `score` に効いていそうな要因を、相関ヒートマップと散布図で調べて。
> 分かったことを、初心者にも分かるように日本語で5行くらいに要約して。グラフの軸は日本語にして。**

### 頼むコツ／注意
- **具体的に**（対象の列・出力の形・日本語で、などを書く）／**小さく頼んで** グラフを見てから次へ
- 出てきたコードは **意味を1つずつ確認**（分からなければそのまま Gemini に質問）
- ⚠️ ①出力は必ず検証（もっともらしく誤る）／②個人情報・機微データは入れない／③アップロードしたファイルはセッション終了で消える

### 前半と見くらべよう
- Gemini の結果は、前半で自分が出した相関・散布図と **同じ結論** になっていますか？
- 違うところ・より詳しいところはどこ？ **最後の解釈は人間の仕事** です。


---
### まとめ
- **前半：自分の手で** データの全体像 → 相関 → 散布図まで EDA を体験できた
- **後半：Gemini に日本語で頼むだけ** でも、同じような分析ノートが作れる（使える環境なら）
- でも **出力は必ず検証**。相関 ≠ 因果。最後の解釈は人間が行う

### 出典・データ
- Data Science Agent in Colab（Gemini）: https://developers.googleblog.com/en/data-science-agent-in-colab-with-gemini/
- データ：CASchools（Rdatasets / R の AER パッケージ由来）https://vincentarelbundock.github.io/Rdatasets/

---
### 🧭 制作クレジット
本ノートブックの **コードは Claude Opus で作成**し、**Claude Fable で検証**しました。最終的な責任は作成者（千葉大学 田川 翔）が負います。生成AIの出力・分析には誤りが含まれることがあります。
